In [107]:
# standard libraries
import io
import warnings
from typing import List, Optional, Union

# third party libararies
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

try:
    import plotly.express as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    PLOTLY_AVAILABLE = True
except ImportError:
    PLOTLY_AVAILABLE = False
    warnings.warn(
        "Plotly is not installed. Run:  pip install plotly\n"
        "Visualization methods will be unavailable until then."
    )

# IPython is always present in Colab
try:
    from IPython.display import display, HTML, FileLink
    IPYTHON_AVAILABLE = True
except ImportError:
    IPYTHON_AVAILABLE = False

# Sentinel null strings recognised during ingestion
_NULL_STRINGS = {"?", "n/a", "na", "null", "none", "nan", "", " ", "--", "N/A",
                 "NULL", "None", "NaN", "NA", "N.A.", "missing", "MISSING"}



# DataInspector class
class DataInspector:
    """
    End-to-end helper for CSV data ingestion, cleaning, feature
    engineering, and statistical visualization.

    Attributes
    ----------
    df : pd.DataFrame
        The working dataset.  Set directly or via ``upload_data()``.

    Examples
    --------
    >>> inspector = DataInspector()
    >>> inspector.upload_data()          # interactive Colab file picker
    >>> inspector.handle_missing_values(strategy='median')
    >>> inspector.remove_duplicates()
    >>> inspector.plot_all_associations_heatmap()
    """

    def __init__(self) -> None:
        self.df: Optional[pd.DataFrame] = None
        self.isFileUploaded = False

    # 1. DATA INGESTION & SANITISATION

    def upload_data(
        self,
        path: Optional[str] = None,
        encoding: str = "utf-8",
        separator: str = ",",
    ) -> None:
        """
        Load a CSV dataset into ``self.df`` via one of three routes —
        chosen automatically based on the ``path`` argument:

        **Route A — explicit file path** (local disk, works everywhere)::

            inspector.upload_data(path="data/titanic.csv")

        **Route B — URL** (any ``http://`` or ``https://`` address)::

            inspector.upload_data(path="https://example.com/data.csv")

        **Route C — interactive Colab file picker** (default, no ``path`` needed)::

            inspector.upload_data()

        All three routes apply the same sanitisation pipeline:
        garbage-string → ``NaN`` replacement and auto numeric coercion.

        Parameters
        ----------
        path : str, optional
            Local file path **or** HTTP/HTTPS URL to a CSV file.
            When omitted, the Colab file-upload widget is shown.
        encoding : str
            File character encoding (default ``'utf-8'``).
            Try ``'latin-1'`` or ``'cp1252'`` if you see decode errors.
        separator : str
            Column delimiter (default ``','``).
            Use ``'\\t'`` for TSV files, ``';'`` for European-style CSVs.

        Examples
        --------
        >>> inspector.upload_data()                          # Colab picker
        >>> inspector.upload_data("titanic.csv")             # local file
        >>> inspector.upload_data("titanic.csv", separator=";")  # semicolon CSV
        >>> inspector.upload_data(                           # remote URL
        ...     "https://raw.githubusercontent.com/datasciencedojo/"
        ...     "datasets/master/titanic.csv"
        ... )
        """
        # ── Route A / B: path or URL was supplied ────────────────────────
        if path is not None:
            import urllib.request

            is_url = path.startswith("http://") or path.startswith("https://")
            source_label = path

            try:
                if is_url:
                    # Download into an in-memory buffer so we never touch disk
                    with urllib.request.urlopen(path) as response:
                        content = response.read()
                    raw = pd.read_csv(
                        io.BytesIO(content),
                        na_values=list(_NULL_STRINGS),
                        keep_default_na=True,
                        encoding=encoding,
                        sep=separator,
                    )
                else:
                    raw = pd.read_csv(
                        path,
                        na_values=list(_NULL_STRINGS),
                        keep_default_na=True,
                        encoding=encoding,
                        sep=separator,
                    )

                self.df = self._auto_convert_types(raw)
                print(
                    f"✅  Loaded '{source_label}'  →  "
                    f"{self.df.shape[0]:,} rows × {self.df.shape[1]:,} columns"
                )

            except FileNotFoundError:
                print(f"❌  File not found: '{path}'\n"
                      f"    Check the path and try again.")
            except Exception as exc:
                print(f"❌  Failed to load '{path}': {exc}")

            return  # always return after handling path/URL

        # ── Route C: no path → Colab interactive picker ──────────────────
        try:
            from google.colab import files
        except ImportError:
            print(
                "⚠️  No path supplied and google.colab is unavailable.\n"
                "    Provide a path or URL:\n"
                "        inspector.upload_data('path/to/file.csv')\n"
                "        inspector.upload_data('https://example.com/data.csv')"
            )
            return

        print("📂  Please select a CSV file using the file picker below…")
        uploaded = files.upload()
        if not uploaded:
            print("No file selected.")
            return

        self.isFileUploaded = True
        filename, content = next(iter(uploaded.items()))
        try:
            raw = pd.read_csv(
                io.BytesIO(content),
                na_values=list(_NULL_STRINGS),
                keep_default_na=True,
                encoding=encoding,
                sep=separator,
            )
            self.df = self._auto_convert_types(raw)
            print(
                f"✅  Loaded '{filename}'  →  "
                f"{self.df.shape[0]:,} rows × {self.df.shape[1]:,} columns"
            )
        except Exception as exc:
            print(f"❌  Could not parse '{filename}': {exc}")

    # ── private helpers ──────────────────────────────────────────────────

    @staticmethod
    def _auto_convert_types(df: pd.DataFrame) -> pd.DataFrame:
        """
        Attempt to coerce object columns to numeric.
        A column is only converted when the coercion does NOT produce an
        entirely-null column (i.e., at least one value parsed successfully).
        """
        df = df.copy()
        for col in df.select_dtypes(include="object").columns:
            converted = pd.to_numeric(df[col], errors="coerce")
            if converted.notna().any():
                df[col] = converted
        return df

    def _require_data(self, method_name: str) -> bool:
        """Emit a clear error if self.df is None and return False."""
        if self.df is None:
            print(f"❌  No data loaded.  Call upload_data() or set "
                  f"inspector.df before calling {method_name}().")
            return False
        return True


    # 2. STRUCTURAL ANALYSIS & CLEANING
    def get_summary(self) -> None:
        """
        Print a high-level structural summary of the dataset:
          • Shape (rows × columns)
          • Column-type breakdown (numeric vs categorical)
          • First 20 rows preview
          • Descriptive statistics for numeric columns
        """
        if not self._require_data("get_summary"):
            return

        df = self.df
        num_cols  = df.select_dtypes(include="number").columns.tolist()
        cat_cols  = df.select_dtypes(exclude="number").columns.tolist()

        print("=" * 60)
        print(f"  DATASET SUMMARY")
        print("=" * 60)
        print(f"  Rows      : {df.shape[0]:,}")
        print(f"  Columns   : {df.shape[1]:,}")
        print(f"  Numeric   : {len(num_cols)}  → {num_cols}")
        print(f"  Categorical: {len(cat_cols)} → {cat_cols}")
        print(f"  Memory    : {df.memory_usage(deep=True).sum() / 1024:.1f} KB")
        print("-" * 60)
        print("\n📋  First 20 rows:")
        display(df.head(20))
        print("\n📊  Numeric statistics:")
        display(df[num_cols].describe().T if num_cols else "No numeric columns.")

    def column_details(self) -> None:
        """
        Display per-column detail: dtype, non-null count, null count,
        null %, unique values, and sample values.
        """
        if not self._require_data("column_details"):
            return

        rows = []
        for col in self.df.columns:
            s = self.df[col]
            null_count = s.isna().sum()
            rows.append({
                "Column"     : col,
                "Dtype"      : str(s.dtype),
                "Non-Null"   : s.notna().sum(),
                "Null Count" : null_count,
                "Null %"     : f"{null_count / len(s) * 100:.1f}%",
                "Unique"     : s.nunique(),
                "Sample"     : s.dropna().iloc[0] if s.notna().any() else "—",
            })
        display(pd.DataFrame(rows).set_index("Column"))

    def get_categorical_summary(self) -> None:
        """
        For each categorical column, display value counts and their
        proportions in a readable table.
        """
        if not self._require_data("get_categorical_summary"):
            return

        cat_cols = self.df.select_dtypes(exclude="number").columns.tolist()
        if not cat_cols:
            print("No categorical columns found.")
            return

        for col in cat_cols:
            vc = self.df[col].value_counts(dropna=False)
            pct = (vc / len(self.df) * 100).round(2)
            summary = pd.DataFrame({"Count": vc, "Percentage %": pct})
            print(f"\n🏷️  {col}  ({self.df[col].dtype})")
            display(summary)

    def show_missing_data(self) -> None:
        """
        Print a table of columns with missing values, sorted by missing
        count descending.  Also shows overall missing-data percentage.
        """
        if not self._require_data("show_missing_data"):
            return

        null_counts = self.df.isna().sum()
        missing = null_counts[null_counts > 0].sort_values(ascending=False)
        if missing.empty:
            print("✅  No missing values found.")
            return

        result = pd.DataFrame({
            "Missing Count": missing,
            "Missing %"    : (missing / len(self.df) * 100).round(2),
        })
        total_cells = self.df.size
        total_missing = missing.sum()
        print(f"⚠️  Missing data: {total_missing:,} / {total_cells:,} cells "
              f"({total_missing / total_cells * 100:.2f}%)\n")
        display(result)

    def handle_missing_values(
        self,
        strategy: str = "mean",
        constant: Union[int, float, str, None] = None,
        columns: Optional[List[str]] = None,
    ) -> None:
        """
        Impute missing values in the dataset.

        Parameters
        ----------
        strategy : str
            One of ``'mean'``, ``'median'``, ``'mode'``, or ``'constant'``.
        constant : scalar, optional
            Fill value when ``strategy='constant'``.
        columns : list of str, optional
            Columns to impute.  Defaults to all columns with nulls.
        """
        if not self._require_data("handle_missing_values"):
            return

        valid = {"mean", "median", "mode", "constant"}
        if strategy not in valid:
            print(f"❌  Unknown strategy '{strategy}'.  Choose from {valid}.")
            return

        targets = columns or self.df.columns[self.df.isna().any()].tolist()
        if not targets:
            print("✅  No missing values to impute.")
            return

        for col in targets:
            if self.df[col].isna().sum() == 0:
                continue
            if strategy == "mean" and pd.api.types.is_numeric_dtype(self.df[col]):
                fill = self.df[col].mean()
            elif strategy == "median" and pd.api.types.is_numeric_dtype(self.df[col]):
                fill = self.df[col].median()
            elif strategy == "mode":
                mode_val = self.df[col].mode()
                fill = mode_val.iloc[0] if not mode_val.empty else np.nan
            elif strategy == "constant":
                fill = constant
            else:
                # Fallback: mode for non-numeric when mean/median requested
                mode_val = self.df[col].mode()
                fill = mode_val.iloc[0] if not mode_val.empty else np.nan

            self.df[col].fillna(fill, inplace=True)
            print(f"  ✔ '{col}' imputed with {strategy} → {fill!r}")

        print(f"\n✅  Imputation complete ({len(targets)} columns processed).")

    def remove_duplicates(self) -> None:
        """
        Remove exact duplicate rows from the dataset and report the
        number of rows dropped.
        """
        if not self._require_data("remove_duplicates"):
            return

        before = len(self.df)
        self.df.drop_duplicates(inplace=True)
        self.df.reset_index(drop=True, inplace=True)
        dropped = before - len(self.df)
        print(f"✅  Removed {dropped:,} duplicate row(s).  "
              f"Dataset now has {len(self.df):,} rows.")

    def handle_outliers(
        self,
        columns: Optional[List[str]] = None,
        find_and_delete: bool = False,
    ) -> None:
        """
        Detect (and optionally remove) outliers using the IQR method.

        Parameters
        ----------
        columns : list of str, optional
            Numeric columns to check.  Defaults to all numeric columns.
        find_and_delete : bool
            If ``True``, rows flagged as outliers are dropped.
            If ``False``, a summary of flagged rows is printed.
        """
        if not self._require_data("handle_outliers"):
            return

        num_cols = (
            columns
            or self.df.select_dtypes(include="number").columns.tolist()
        )
        outlier_mask = pd.Series(False, index=self.df.index)

        for col in num_cols:
            if col not in self.df.columns:
                print(f"⚠️  Column '{col}' not found — skipped.")
                continue
            q1  = self.df[col].quantile(0.25)
            q3  = self.df[col].quantile(0.75)
            iqr = q3 - q1
            lower = q1 - 1.5 * iqr
            upper = q3 + 1.5 * iqr
            col_mask = (self.df[col] < lower) | (self.df[col] > upper)
            count = col_mask.sum()
            print(f"  📌 '{col}': {count:,} outlier(s)  "
                  f"[bounds: {lower:.3f} — {upper:.3f}]")
            outlier_mask |= col_mask

        total_flagged = outlier_mask.sum()
        if find_and_delete:
            self.df = self.df[~outlier_mask].reset_index(drop=True)
            print(f"\n✅  Deleted {total_flagged:,} outlier row(s).  "
                  f"Dataset now has {len(self.df):,} rows.")
        else:
            print(f"\n⚠️  {total_flagged:,} row(s) flagged as outliers.  "
                  f"Pass find_and_delete=True to remove them.")

    def delete_rows(self) -> None:
        """
        Interactively delete rows by index.

        Prompts the user to enter comma-separated row indices to drop.
        """
        if not self._require_data("delete_rows"):
            return

        print("Current row indices:", list(self.df.index[:20]),
              "..." if len(self.df) > 20 else "")
        raw = input("Enter comma-separated row indices to delete: ").strip()
        if not raw:
            print("No indices provided — nothing deleted.")
            return
        try:
            indices = [int(i.strip()) for i in raw.split(",")]
            valid   = [i for i in indices if i in self.df.index]
            self.df.drop(index=valid, inplace=True)
            self.df.reset_index(drop=True, inplace=True)
            print(f"✅  Deleted {len(valid)} row(s).")
        except ValueError:
            print("❌  Invalid input.  Enter integer indices separated by commas.")

    def delete_columns(self) -> None:
        """
        Interactively delete columns by name.

        Prompts the user to enter comma-separated column names to drop.
        """
        if not self._require_data("delete_columns"):
            return

        print("Columns:", self.df.columns.tolist())
        raw = input("Enter comma-separated column names to delete: ").strip()
        if not raw:
            print("No columns provided — nothing deleted.")
            return
        names  = [c.strip() for c in raw.split(",")]
        valid  = [c for c in names if c in self.df.columns]
        invalid = [c for c in names if c not in self.df.columns]
        if invalid:
            print(f"⚠️  Not found (skipped): {invalid}")
        self.df.drop(columns=valid, inplace=True)
        print(f"✅  Deleted columns: {valid}")

    # 3. FEATURE ENGINEERING PREPARATION
    def extract_normalized_numeric_data(
        self, method: str = "minmax"
    ) -> Optional[pd.DataFrame]:
        """
        Scale numeric columns and return the result as a new DataFrame.

        Parameters
        ----------
        method : str
            ``'minmax'``   → scales each feature to [0, 1].
            ``'standard'`` → zero mean, unit variance (Z-score).
            ``'robust'``   → uses median and IQR (robust to outliers).

        Returns
        -------
        pd.DataFrame or None
            Scaled DataFrame, or ``None`` if no numeric columns exist.
        """
        if not self._require_data("extract_normalized_numeric_data"):
            return None

        num_df = self.df.select_dtypes(include="number").copy()
        if num_df.empty:
            print("No numeric columns to scale.")
            return None

        scalers = {
            "minmax"  : MinMaxScaler(),
            "standard": StandardScaler(),
            "robust"  : RobustScaler(),
        }
        if method not in scalers:
            print(f"❌  Unknown method '{method}'.  Choose from {list(scalers)}.")
            return None

        scaler = scalers[method]
        scaled = pd.DataFrame(
            scaler.fit_transform(num_df),
            columns=num_df.columns,
            index=num_df.index,
        )
        print(f"✅  Numeric data scaled using '{method}' method "
              f"({len(scaled.columns)} columns).")
        return scaled

    def extract_normalized_categorical_data(
        self, method: str = "onehot"
    ) -> Optional[pd.DataFrame]:
        """
        Encode categorical columns and return the result as a new DataFrame.

        Parameters
        ----------
        method : str
            ``'onehot'``  → binary indicator columns for each category.
            ``'ordinal'`` → integer label encoding.
            ``'uniform'`` → ordinal codes scaled to [0, 1].

        Returns
        -------
        pd.DataFrame or None
        """
        if not self._require_data("extract_normalized_categorical_data"):
            return None

        cat_df = self.df.select_dtypes(exclude="number").copy()
        if cat_df.empty:
            print("No categorical columns to encode.")
            return None

        if method == "onehot":
            encoded = pd.get_dummies(cat_df, drop_first=False)

        elif method == "ordinal":
            encoded = cat_df.copy()
            for col in cat_df.columns:
                encoded[col] = pd.Categorical(cat_df[col]).codes.astype(float)
                encoded[col].replace(-1, np.nan, inplace=True)

        elif method == "uniform":
            encoded = cat_df.copy()
            for col in cat_df.columns:
                codes = pd.Categorical(cat_df[col]).codes.astype(float)
                codes[codes == -1] = np.nan
                col_min, col_max = np.nanmin(codes), np.nanmax(codes)
                if col_max > col_min:
                    codes = (codes - col_min) / (col_max - col_min)
                encoded[col] = codes

        else:
            print(f"❌  Unknown method '{method}'.  "
                  "Choose from 'onehot', 'ordinal', 'uniform'.")
            return None

        print(f"✅  Categorical data encoded using '{method}' method "
              f"({len(encoded.columns)} output columns).")
        return encoded

    def create_normalized_data_df(
        self,
        numeric_method : str = "minmax",
        categorical_method: str = "onehot",
    ) -> Optional[pd.DataFrame]:
        """
        Build a single, ML-ready DataFrame by merging scaled numeric columns
        with encoded categorical columns.

        Parameters
        ----------
        numeric_method : str
            Passed to ``extract_normalized_numeric_data()``.
        categorical_method : str
            Passed to ``extract_normalized_categorical_data()``.

        Returns
        -------
        pd.DataFrame or None
        """
        num_scaled = self.extract_normalized_numeric_data(method=numeric_method)
        cat_encoded = self.extract_normalized_categorical_data(
            method=categorical_method
        )

        parts = [p for p in [num_scaled, cat_encoded] if p is not None]
        if not parts:
            print("No data to merge.")
            return None

        merged = pd.concat(parts, axis=1)
        print(f"✅  Merged DataFrame: {merged.shape[0]:,} rows × "
              f"{merged.shape[1]:,} columns.")
        return merged


    # 4. INTERACTIVE VISUALISATION
    @staticmethod
    def _check_plotly() -> bool:
        if not PLOTLY_AVAILABLE:
            print("❌  Plotly is required.  Run: pip install plotly")
            return False
        return True

    def plot_numerical(
        self,
        column_names: Optional[List[str]] = None,
        max_cols: int = 3,
    ) -> None:
        """
        For each specified numeric column, render a 3-panel subplot:
        horizontal violin, scatter (index vs value), and histogram.

        Parameters
        ----------
        column_names : list of str, optional
            Columns to visualise.  Defaults to all numeric columns.
        max_cols : int
            Number of column panels per row (default 3).
        """
        if not self._require_data("plot_numerical") or not self._check_plotly():
            return

        cols = column_names or self.df.select_dtypes(include="number").columns.tolist()
        cols = [c for c in cols if c in self.df.columns]
        if not cols:
            print("No valid numeric columns to plot.")
            return

        for col in cols:
            series = self.df[col].dropna()
            fig = make_subplots(
                rows=1, cols=3,
                subplot_titles=(
                    f"Violin — {col}",
                    f"Scatter — {col}",
                    f"Histogram — {col}",
                ),
            )
            # Panel 1 — horizontal violin
            fig.add_trace(
                go.Violin(x=series, name=col, box_visible=True,
                          meanline_visible=True, orientation="h"),
                row=1, col=1,
            )
            # Panel 2 — scatter (index vs value)
            fig.add_trace(
                go.Scatter(
                    x=series.index, y=series, mode="markers",
                    marker=dict(size=3, opacity=0.5),
                    name=col,
                ),
                row=1, col=2,
            )
            # Panel 3 — histogram
            fig.add_trace(
                go.Histogram(x=series, nbinsx=30, name=col),
                row=1, col=3,
            )
            fig.update_layout(
                title_text=f"Distribution of '{col}'",
                height=350,
                showlegend=False,
                template="plotly_white",
            )
            fig.show()

    def plot_categorical(
        self, column_names: Optional[List[str]] = None
    ) -> None:
        """
        For each categorical column, display a bar chart showing raw
        counts and percentage labels.

        Parameters
        ----------
        column_names : list of str, optional
            Columns to visualise.  Defaults to all categorical columns.
        """
        if not self._require_data("plot_categorical") or not self._check_plotly():
            return

        cols = (
            column_names
            or self.df.select_dtypes(exclude="number").columns.tolist()
        )
        cols = [c for c in cols if c in self.df.columns]
        if not cols:
            print("No valid categorical columns to plot.")
            return

        for col in cols:
            vc  = self.df[col].value_counts(dropna=False)
            pct = (vc / vc.sum() * 100).round(1)
            fig = go.Figure(go.Bar(
                x=vc.index.astype(str),
                y=vc.values,
                text=[f"{p}%" for p in pct],
                textposition="outside",
                marker_color="steelblue",
            ))
            fig.update_layout(
                title=f"Frequency — '{col}'",
                xaxis_title=col,
                yaxis_title="Count",
                template="plotly_white",
                height=400,
            )
            fig.show()

    def plot_relationship(self, col_x: str, col_y: str) -> None:
        """
        Auto-detect the types of two columns and choose the most
        appropriate chart:

        * **Num–Num**   → Scatter with OLS trendline.
        * **Cat–Num**   → Box plot with strip overlay.
        * **Num–Cat**   → Box plot (axes flipped).
        * **Cat–Cat**   → Grouped bar chart.

        Parameters
        ----------
        col_x, col_y : str
            Column names in ``self.df``.
        """
        if not self._require_data("plot_relationship") or not self._check_plotly():
            return

        for col in (col_x, col_y):
            if col not in self.df.columns:
                print(f"❌  Column '{col}' not found.")
                return

        x_num = pd.api.types.is_numeric_dtype(self.df[col_x])
        y_num = pd.api.types.is_numeric_dtype(self.df[col_y])

        if x_num and y_num:
            fig = px.scatter(
                self.df, x=col_x, y=col_y,
                trendline="ols",
                title=f"{col_x} vs {col_y}  (Num × Num)",
                template="plotly_white",
                opacity=0.6,
            )
        elif not x_num and y_num:
            fig = px.box(
                self.df, x=col_x, y=col_y,
                points="all",
                title=f"{col_y} by {col_x}  (Cat × Num)",
                template="plotly_white",
            )
        elif x_num and not y_num:
            fig = px.box(
                self.df, x=col_y, y=col_x,
                points="all",
                title=f"{col_x} by {col_y}  (Num × Cat)",
                template="plotly_white",
            )
        else:
            ct = (
                pd.crosstab(self.df[col_x], self.df[col_y])
                .reset_index()
                .melt(id_vars=col_x)
            )
            fig = px.bar(
                ct, x=col_x, y="value", color=col_y,
                barmode="group",
                title=f"{col_x} vs {col_y}  (Cat × Cat)",
                template="plotly_white",
            )

        fig.update_layout(height=500)
        fig.show()


    # 5. STATISTICAL ASSOCIATION HEATMAPS
    def plot_numerical_correlation(self) -> None:
        """
        Display a Pearson correlation heatmap for all numeric columns.
        """
        if not self._require_data("plot_numerical_correlation") or not self._check_plotly():
            return

        num_df = self.df.select_dtypes(include="number")
        if num_df.shape[1] < 2:
            print("Need at least 2 numeric columns.")
            return

        corr = num_df.corr()
        fig = px.imshow(
            corr,
            text_auto=".2f",
            color_continuous_scale="RdBu_r",
            zmin=-1, zmax=1,
            title="Pearson Correlation — Numeric Columns",
            template="plotly_white",
        )
        fig.update_layout(height=600)
        fig.show()

    def plot_categorical_correlation(self) -> None:
        """
        Display a Cramér's V heatmap for all categorical columns.
        Cramér's V ranges from 0 (no association) to 1 (perfect association).
        """
        if not self._require_data("plot_categorical_correlation") or not self._check_plotly():
            return

        cat_cols = self.df.select_dtypes(exclude="number").columns.tolist()
        if len(cat_cols) < 2:
            print("Need at least 2 categorical columns.")
            return

        def cramers_v(x: pd.Series, y: pd.Series) -> float:
            ct = pd.crosstab(x, y)
            chi2 = stats.chi2_contingency(ct, correction=False)[0]
            n = ct.sum().sum()
            phi2 = chi2 / n
            r, k = ct.shape
            return np.sqrt(phi2 / max(min(k - 1, r - 1), 1))

        mat = pd.DataFrame(
            np.zeros((len(cat_cols), len(cat_cols))),
            index=cat_cols, columns=cat_cols,
        )
        for i, ci in enumerate(cat_cols):
            for j, cj in enumerate(cat_cols):
                mat.iloc[i, j] = (
                    1.0 if i == j
                    else cramers_v(self.df[ci].dropna(), self.df[cj].dropna())
                )

        fig = px.imshow(
            mat,
            text_auto=".2f",
            color_continuous_scale="Blues",
            zmin=0, zmax=1,
            title="Cramér's V — Categorical Columns",
            template="plotly_white",
        )
        fig.update_layout(height=600)
        fig.show()

    def plot_all_associations_heatmap(self) -> None:
        """
        Render a unified association heatmap covering all column types:

        * **Num–Num**  : Pearson's r
        * **Cat–Cat**  : Cramér's V
        * **Num–Cat**  : Eta coefficient (ANOVA / correlation ratio)
        * **Cat–Num**  : same as Num–Cat (symmetric)

        Values are on a shared [0, 1] scale (absolute values for Pearson).
        """
        if not self._require_data("plot_all_associations_heatmap") or not self._check_plotly():
            return

        df = self.df
        cols = df.columns.tolist()
        n    = len(cols)

        def is_num(c: str) -> bool:
            return pd.api.types.is_numeric_dtype(df[c])

        def pearson(ci: str, cj: str) -> float:
            return abs(df[[ci, cj]].dropna().corr().iloc[0, 1])

        def cramers_v(ci: str, cj: str) -> float:
            ct = pd.crosstab(df[ci], df[cj])
            chi2 = stats.chi2_contingency(ct, correction=False)[0]
            phi2 = chi2 / ct.sum().sum()
            r, k = ct.shape
            return np.sqrt(phi2 / max(min(k - 1, r - 1), 1))

        def eta(num_col: str, cat_col: str) -> float:
            """Correlation ratio (η): proportion of variance explained."""
            groups = [
                grp[num_col].dropna().values
                for _, grp in df.groupby(cat_col)
                if grp[num_col].notna().any()
            ]
            if len(groups) < 2:
                return 0.0
            all_vals = df[num_col].dropna()
            grand_mean = all_vals.mean()
            ss_between = sum(
                len(g) * (g.mean() - grand_mean) ** 2 for g in groups
            )
            ss_total = ((all_vals - grand_mean) ** 2).sum()
            return np.sqrt(ss_between / ss_total) if ss_total > 0 else 0.0

        mat = np.zeros((n, n))
        for i, ci in enumerate(cols):
            for j, cj in enumerate(cols):
                if i == j:
                    mat[i, j] = 1.0
                elif is_num(ci) and is_num(cj):
                    mat[i, j] = pearson(ci, cj)
                elif not is_num(ci) and not is_num(cj):
                    mat[i, j] = cramers_v(ci, cj)
                elif is_num(ci) and not is_num(cj):
                    mat[i, j] = eta(ci, cj)
                else:
                    mat[i, j] = eta(cj, ci)

        assoc_df = pd.DataFrame(mat, index=cols, columns=cols)
        fig = px.imshow(
            assoc_df,
            text_auto=".2f",
            color_continuous_scale="Viridis",
            zmin=0, zmax=1,
            title="Unified Association Heatmap  (|Pearson r| / Cramér's V / η)",
            template="plotly_white",
        )
        fig.update_layout(height=max(500, 60 * n))
        fig.show()


    # Helper: delegate PlottingMethods methods onto the inspector
    def display_image(self, result: dict) -> None:
        """Render an HTML Plotly figure returned by PlottingMethods."""
        if not IPYTHON_AVAILABLE:
            print("IPython display not available.")
            return
        html = result.get("html")
        if html:
            display(HTML(html))
        else:
            print(f"⚠️  Status: {result.get('status', 'unknown error')}")


#PlottingMethods class
class PlottingMethods:
    """
    Standalone, modular chart generators.

    Each method accepts either a raw ``pd.DataFrame`` via the ``data``
    parameter or JSON-serializable data.  Every method returns a dict::

        {
            "status" : "success" | "error",
            "html"   : "<div>…plotly figure…</div>",   # on success
            "message": "error description",             # on error
        }

    Use ``display_image(result)`` to render the figure in Colab.

    Examples
    --------
    >>> plt = PlottingMethods()
    >>> result = plt.plot_pie_chart(names='team', values='incentive', data=df)
    >>> plt.display_image(result)
    """

    # ── private helpers ──────────────────────────────────────────────────

    @staticmethod
    def _to_df(data) -> Optional[pd.DataFrame]:
        """
        Coerce ``data`` to a ``pd.DataFrame``.
        Accepts DataFrame, list-of-dicts, or JSON string.
        Returns None on failure.
        """
        if isinstance(data, pd.DataFrame):
            return data
        if isinstance(data, list):
            try:
                return pd.DataFrame(data)
            except Exception:
                return None
        if isinstance(data, str):
            try:
                import json
                return pd.DataFrame(json.loads(data))
            except Exception:
                return None
        return None

    @staticmethod
    def _wrap(fig) -> dict:
        """Serialize a Plotly figure to an HTML string."""
        return {
            "status": "success",
            "html"  : fig.to_html(full_html=False, include_plotlyjs="cdn"),
        }

    @staticmethod
    def _err(message: str) -> dict:
        return {"status": "error", "message": message}

    def display_image(self, result: dict) -> None:
        """
        Render the HTML Plotly figure from a PlottingMethods result dict.

        Parameters
        ----------
        result : dict
            Return value of any plot method.
        """
        if not IPYTHON_AVAILABLE:
            print("IPython not available outside Colab / Jupyter.")
            return
        if result.get("status") != "success":
            print(f"❌  {result.get('message', 'Unknown error.')}")
            return
        display(HTML(result["html"]))

    # ── chart methods ─────────────────────────────────────────────────────

    def plot_bar_chart(
        self,
        x: str,
        y: str,
        data,
        color: Optional[str] = None,
        barmode: str = "group",
        title: Optional[str] = None,
    ) -> dict:
        """
        Generate a grouped or stacked bar chart.

        Parameters
        ----------
        x, y : str
            Column names for the x-axis category and y-axis value.
        data : DataFrame or list-of-dicts
            Source data.
        color : str, optional
            Column used to split bars into colour groups.
        barmode : str
            ``'group'`` (default) or ``'stack'``.
        title : str, optional
            Chart title.
        """
        df = self._to_df(data)
        if df is None:
            return self._err("Could not parse 'data' as a DataFrame.")
        for col in (c for c in [x, y, color] if c):
            if col not in df.columns:
                return self._err(f"Column '{col}' not found.")
        fig = px.bar(
            df, x=x, y=y, color=color, barmode=barmode,
            title=title or f"{y} by {x}",
            template="plotly_white",
        )
        return self._wrap(fig)

    def plot_pie_chart(
        self,
        names: str,
        values: str,
        data,
        hole: float = 0.0,
        title: Optional[str] = None,
    ) -> dict:
        """
        Generate a pie (or donut) chart.

        Parameters
        ----------
        names, values : str
            Column names for slice labels and slice sizes.
        data : DataFrame or list-of-dicts
        hole : float
            Donut hole size in [0, 1].  ``0.0`` = solid pie.
        title : str, optional
        """
        df = self._to_df(data)
        if df is None:
            return self._err("Could not parse 'data' as a DataFrame.")
        for col in (names, values):
            if col not in df.columns:
                return self._err(f"Column '{col}' not found.")
        agg = df.groupby(names, as_index=False)[values].sum()
        fig = go.Figure(go.Pie(
            labels=agg[names],
            values=agg[values],
            hole=hole,
            textinfo="label+percent",
        ))
        fig.update_layout(
            title=title or f"{values} split by {names}",
            template="plotly_white",
        )
        return self._wrap(fig)

    def plot_histogram(
        self,
        x: str,
        data,
        bins: Optional[List[float]] = None,
        title: Optional[str] = None,
    ) -> dict:
        """
        Generate a histogram with optional custom bin boundaries.

        Parameters
        ----------
        x : str
            Column to plot.
        data : DataFrame or list-of-dicts
        bins : list of float, optional
            Explicit bin edges, e.g. ``[0, 18, 35, 60, 100]``.
        title : str, optional
        """
        df = self._to_df(data)
        if df is None:
            return self._err("Could not parse 'data' as a DataFrame.")
        if x not in df.columns:
            return self._err(f"Column '{x}' not found.")

        if bins:
            fig = go.Figure(go.Histogram(
                x=df[x],
                xbins=dict(
                    start=bins[0],
                    end=bins[-1],
                    size=(bins[-1] - bins[0]) / (len(bins) - 1),
                ),
                autobinx=False,
            ))
        else:
            fig = px.histogram(df, x=x, nbins=30, template="plotly_white")

        fig.update_layout(
            title=title or f"Distribution of {x}",
            xaxis_title=x,
            yaxis_title="Count",
            template="plotly_white",
        )
        return self._wrap(fig)

    def plot_heat_map(
        self,
        values: str,
        index: str,
        columns: str,
        data,
        aggregade_method: str = "mean",
        title: Optional[str] = None,
    ) -> dict:
        """
        Generate a pivot-table heatmap.

        Parameters
        ----------
        values : str     Column whose values fill the cells.
        index  : str     Row labels column.
        columns: str     Column labels column.
        data           : DataFrame or list-of-dicts.
        aggregade_method : str  Aggregation function (``'mean'``, ``'sum'``, etc.).
        title  : str, optional
        """
        df = self._to_df(data)
        if df is None:
            return self._err("Could not parse 'data' as a DataFrame.")
        for col in (values, index, columns):
            if col not in df.columns:
                return self._err(f"Column '{col}' not found.")
        pivot = df.pivot_table(
            values=values, index=index, columns=columns,
            aggfunc=aggregade_method,
        )
        fig = px.imshow(
            pivot,
            text_auto=".2f",
            color_continuous_scale="YlOrRd",
            title=title or f"{values} ({aggregade_method}) by {index} × {columns}",
            template="plotly_white",
        )
        return self._wrap(fig)

    def plot_sankey_diagram(
        self,
        source_column: str,
        target_column: str,
        values: str,
        data,
        title: Optional[str] = None,
    ) -> dict:
        """
        Generate a Sankey (flow) diagram.

        Parameters
        ----------
        source_column, target_column : str
            Columns defining the flow direction.
        values : str
            Column used to size each flow link.
        data : DataFrame or list-of-dicts
        title : str, optional
        """
        df = self._to_df(data)
        if df is None:
            return self._err("Could not parse 'data' as a DataFrame.")
        for col in (source_column, target_column, values):
            if col not in df.columns:
                return self._err(f"Column '{col}' not found.")

        agg = df.groupby([source_column, target_column], as_index=False)[values].sum()
        all_nodes = pd.unique(
            agg[[source_column, target_column]].values.ravel()
        ).tolist()
        node_idx  = {n: i for i, n in enumerate(all_nodes)}

        fig = go.Figure(go.Sankey(
            node=dict(label=all_nodes, pad=15, thickness=20),
            link=dict(
                source=[node_idx[s] for s in agg[source_column]],
                target=[node_idx[t] for t in agg[target_column]],
                value =agg[values].tolist(),
            ),
        ))
        fig.update_layout(
            title=title or f"Flow: {source_column} → {target_column}",
            template="plotly_white",
            height=500,
        )
        return self._wrap(fig)

    def plot_simple_sunburst_graph(
        self,
        path: List[str],
        values: str,
        data,
        title: Optional[str] = None,
    ) -> dict:
        """
        Generate a sunburst (hierarchical pie) chart.

        Parameters
        ----------
        path : list of str
            Ordered list of categorical columns defining the hierarchy
            (outer → inner), e.g. ``['department', 'quarter']``.
        values : str
            Numeric column used to size each sector.
        data : DataFrame or list-of-dicts
        title : str, optional
        """
        df = self._to_df(data)
        if df is None:
            return self._err("Could not parse 'data' as a DataFrame.")
        for col in (*path, values):
            if col not in df.columns:
                return self._err(f"Column '{col}' not found.")
        fig = px.sunburst(
            df, path=path, values=values,
            title=title or " / ".join(path),
            template="plotly_white",
        )
        return self._wrap(fig)

    def get_methods_info(self) -> dict:
        """
        Return metadata about every available plotting method.

        Returns
        -------
        dict
            ``{"response": [{"Method": ..., "Description": ..., "Key Parameters": ...}, ...]}``
        """
        info = [
            {
                "Method"        : "plot_bar_chart",
                "Description"   : "Grouped or stacked bar chart.",
                "Key Parameters": "x, y, data, color, barmode",
            },
            {
                "Method"        : "plot_pie_chart",
                "Description"   : "Pie or donut chart.",
                "Key Parameters": "names, values, data, hole",
            },
            {
                "Method"        : "plot_histogram",
                "Description"   : "Distribution histogram with optional custom bins.",
                "Key Parameters": "x, data, bins",
            },
            {
                "Method"        : "plot_heat_map",
                "Description"   : "Pivot-table heatmap.",
                "Key Parameters": "values, index, columns, data, aggregade_method",
            },
            {
                "Method"        : "plot_sankey_diagram",
                "Description"   : "Flow diagram between two categorical columns.",
                "Key Parameters": "source_column, target_column, values, data",
            },
            {
                "Method"        : "plot_simple_sunburst_graph",
                "Description"   : "Hierarchical sunburst chart.",
                "Key Parameters": "path, values, data",
            },
        ]
        return {"response": info}

In [108]:
inspector = DataInspector()

In [109]:
# Install + import
!pip install "git+https://github.com/mugalan/data-analysis-tool.git"  # if needed
from data_analysis import DataInspector, PlottingMethods

import pandas as pd

  Cloning https://github.com/mugalan/data-analysis-tool.git to /tmp/pip-req-build-r0ivmu7o
  Running command git clone --filter=blob:none --quiet https://github.com/mugalan/data-analysis-tool.git /tmp/pip-req-build-r0ivmu7o
  Resolved https://github.com/mugalan/data-analysis-tool.git to commit 013fa81601516dddb01c477348242d865f5b511e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


#Data Import

Custom Data Upload (CSV file type)

In [110]:
inspector.upload_data()

📂  Please select a CSV file using the file picker below…


No file selected.


If no custom files are uplaoded, upload Titanic.csv

In [111]:

# Full flow: Titanic
if not inspector.isFileUploaded:
  inspector.df = pd.read_csv("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv")
else:
  print("Custom CSV file has uploaded")


# 1. Dataset Overview & Structure Summary

In [112]:
inspector.get_summary()


  DATASET SUMMARY
  Rows      : 891
  Columns   : 12
  Numeric   : 7  → ['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
  Categorical: 5 → ['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked']
  Memory    : 285.6 KB
------------------------------------------------------------

📋  First 20 rows:


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C



📊  Numeric statistics:


,count,mean,std,min,25%,50%,75%,max
PassengerId,891.0,446.000000,257.353842,1.00,223.5000,446.0000,668.5,891.0000
Survived,891.0,0.383838,0.486592,0.00,0.0000,0.0000,1.0,1.0000
Pclass,891.0,2.308642,0.836071,1.00,2.0000,3.0000,3.0,3.0000
Age,714.0,29.699118,14.526497,0.42,20.1250,28.0000,38.0,80.0000
SibSp,891.0,0.523008,1.102743,0.00,0.0000,0.0000,1.0,8.0000
Parch,891.0,0.381594,0.806057,0.00,0.0000,0.0000,0.0,6.0000
Fare,891.0,32.204208,49.693429,0.00,7.9104,14.4542,31.0,512.3292


# 2. Missing Value Imputation (Median Strategy)

In [113]:
inspector.handle_missing_values(strategy='median')

  ✔ 'Age' imputed with median → 28.0
  ✔ 'Cabin' imputed with median → 'B96 B98'
  ✔ 'Embarked' imputed with median → 'S'

✅  Imputation complete (3 columns processed).


/tmp/ipykernel_806/1339141252.py:358: FutureWarning:

A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.





# 3. Duplicate Row Removal

In [114]:
inspector.remove_duplicates()

✅  Removed 0 duplicate row(s).  Dataset now has 891 rows.


# 4. Numerical Distribution Analysis — Age, Fare & SibSp

In [115]:
inspector.plot_numerical(['Age', 'Fare', 'SibSp'])

# 5. Relationship Analysis — Passenger Class vs Fare (Cat × Num)

In [116]:
inspector.plot_relationship('Pclass', 'Fare')       # Cat × Num → box plot

# 6. Unified Association Heatmap — All Column Types

In [117]:
inspector.plot_all_associations_heatmap()

# 7. Normalized & Encoded ML-Ready DataFrame


In [118]:
final = inspector.create_normalized_data_df()

✅  Numeric data scaled using 'minmax' method (7 columns).
✅  Categorical data encoded using 'onehot' method (1724 output columns).
✅  Merged DataFrame: 891 rows × 1,731 columns.
